# [SITCOM-2110] - M1M3 Force actuator following error analysis for a whole night

We would like to identify which actuators have larger following errors

Given a day_obs, this notebook will show what are the actuators hitting the highest following errors (positive and negative, both for primary and secondary actuators) as well as the what are these errors as a function of time during the day_obs.

Another cell allows to compute the FFT of the force actuator following errors, for detailed analysis.

[SITCOM-2110]: https://rubinobs.atlassian.net/browse/SITCOM-2110

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta
from pathlib import Path

from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState
from lsst.summit.utils.efdUtils import EfdClient, getEfdData, makeEfdClient
from lsst.sitcom.vandv import m1m3
from lsst.ts.xml.tables.m1m3 import FATable
#from lsst.ts.xml.tables.m1m3 import FATable, FAIndex, force_actuator_from_id, actuator_id_to_index
from lsst.ts.xml.enums.MTM1M3 import DetailedStates

from collections import defaultdict
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, TapTool, Legend, ColumnDataSource, Range1d
from bokeh.layouts import column
from bokeh.io import output_notebook

from scipy.signal import find_peaks

In [ ]:
client = makeEfdClient()

In [ ]:
N_ACTUATORS = len(FATable)
N_SECONDARY = 112

In [ ]:
# this array allows identifying the actuator id 
# given an index from the secondary following errors from 1 to N_SECONDARY
secondary_actuator_id = np.array([])
for i in range(N_ACTUATORS):
    if FATable[i].s_index is not None:
        secondary_actuator_id = np.append(secondary_actuator_id,FATable[i].actuator_id)
secondary_actuator_id = secondary_actuator_id.astype(int)

### Settings

In [ ]:
## Insert here the day_obs of interest
day_obs = 20250518
# other settings
# number of seconds before and after start of slew to grab information
pre_slew_delta = TimeDelta(3, format='sec')
post_slew_delta = TimeDelta(5, format='sec')
# number of top actuators with highest errors to identify per slew
nb_per_slew = 5
# start and finish for frequency analysis
# change the times as appropriate (this is independent of day_obs)
t_start_freq_analysis = Time("2025-05-19 10:27:32.431173",scale="utc") 
t_end_freq_analysis = Time("2025-05-19 10:28:18.061117",scale="utc")

### retrieve data from EFD

#### read slews for day_obs

In [ ]:
# Select data from a given date
eventMaker = TMAEventMaker()
events = eventMaker.getEvents(day_obs)

# Get lists of slew and track events
slews = [e for e in events if e.type == TMAState.SLEWING]
tracks = [e for e in events if e.type == TMAState.TRACKING]
print(f"There are {len(events)} events")
print(f"Found {len(slews)} slews and {len(tracks)} tracks")

#### loop through all slews

In [ ]:
primary_FA_error = [f"primaryCylinderFollowingError{i}" for i in range(N_ACTUATORS)]
secondary_FA_error = [f"secondaryCylinderFollowingError{i}" for i in range(N_SECONDARY)] 
max_actuator_ids_primary = []
min_actuator_ids_primary = []
max_values_primary = []
min_values_primary = []
max_actuator_ids_secondary = []
min_actuator_ids_secondary = []
max_values_secondary = []
min_values_secondary = []

timestamps_primary = []
timestamps_shifted_primary = []
timestamps_secondary = []
timestamps_shifted_secondary = []

aid = np.empty(N_ACTUATORS)
for j in range(N_ACTUATORS):
    aid[j] = FATable[j].actuator_id
maxbin = int(np.max(aid))
minbin = int(np.min(aid))

for i, slew in enumerate(slews):
    if i%10 == 0:
        print(i)
    df_primary_FA_error = getEfdData(
        client,"lsst.sal.MTM1M3.forceActuatorData", 
        columns=primary_FA_error,
        prePadding = pre_slew_delta,
        postPadding = post_slew_delta,
        begin=slew.begin, 
        end=slew.end,
    )
    df_secondary_FA_error = getEfdData( #more convenient to fill a different data frame
        client,"lsst.sal.MTM1M3.forceActuatorData", 
        columns=secondary_FA_error,
        prePadding = pre_slew_delta,
        postPadding = post_slew_delta,
        begin=slew.begin, 
        end=slew.end,
    )    
    if (df_primary_FA_error.empty) or (df_secondary_FA_error.empty):
        print(f"Slew {i} returns an empty dataframe")
        continue
    max_val_primary = df_primary_FA_error.max().nlargest(nb_per_slew)
    max_n_primary = max_val_primary.index
    max_actuators_primary = [df_primary_FA_error.max().index.get_loc(idx) for idx in max_n_primary]
    for k,max_actuator_primary in enumerate(max_actuators_primary):
        timestamps_primary.append((slew.begin + TimeDelta(15,format='sec')).to_datetime())
        timestamps_shifted_primary.append((slew.begin + TimeDelta(20,format='sec')).to_datetime())
        max_actuator_ids_primary.append(FATable[max_actuator_primary].actuator_id)
        max_values_primary.append(max_val_primary.iloc[k])
    min_val_primary = df_primary_FA_error.min().nsmallest(nb_per_slew)
    min_n_primary = min_val_primary.index
    min_actuators_primary = [df_primary_FA_error.min().index.get_loc(idx) for idx in min_n_primary]
    for k,min_actuator_primary in enumerate (min_actuators_primary):
        min_actuator_ids_primary.append(FATable[min_actuator_primary].actuator_id)
        min_values_primary.append(abs(min_val_primary.iloc[k]))

    max_val_secondary = df_secondary_FA_error.max().nlargest(nb_per_slew)
    max_n_secondary = max_val_secondary.index
    max_actuators_secondary = [df_secondary_FA_error.max().index.get_loc(idx) for idx in max_n_secondary]
    for k,max_actuator_secondary in enumerate(max_actuators_secondary):
        timestamps_secondary.append((slew.begin + TimeDelta(15,format='sec')).to_datetime())
        timestamps_shifted_secondary.append((slew.begin + TimeDelta(20,format='sec')).to_datetime())
        max_actuator_ids_secondary.append(secondary_actuator_id[max_actuator_secondary])
        max_values_secondary.append(max_val_secondary.iloc[k])
    min_val_secondary = df_secondary_FA_error.min().nsmallest(nb_per_slew)
    min_n_secondary = min_val_secondary.index
    min_actuators_secondary = [df_secondary_FA_error.min().index.get_loc(idx) for idx in min_n_secondary]
    for k,min_actuator_secondary in enumerate (min_actuators_secondary):
        min_actuator_ids_secondary.append(secondary_actuator_id[min_actuator_secondary])
        min_values_secondary.append(abs(min_val_secondary.iloc[k]))



### Plot largest and lowest values as a function of time  

Use nb_per_slew points for each slew

In [ ]:
fig, ax = plt.subplots(2, 1, dpi=125, figsize=(8, 4))
ax[0].scatter(timestamps_primary, max_values_primary, color='red', marker=".",label='Largest positive')
ax[0].scatter(timestamps_primary, min_values_primary, color='blue', marker=".",label='Largest negative')
ax[0].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax[0].set_ylabel("FA primary \n following errors \n (N)")
ax[1].scatter(timestamps_secondary, max_values_secondary, color='red', marker=".")
ax[1].scatter(timestamps_secondary, min_values_secondary, color='blue', marker=".")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax[1].set_ylabel("FA secondary \n following errors \n (N)")
ax[1].set_xlabel("UTC")
fig.autofmt_xdate()
fig.suptitle(f"Largest positive and largest negative (absolute) values \n per slew taking top {nb_per_slew} values. day_obs: {day_obs}")
fig.legend()
fig.tight_layout()

### Plot histogram of top actuator ids with largest FA errors

In [ ]:
maxhist = plt.hist(
    max_actuator_ids_primary, bins=maxbin-minbin, range=[minbin, maxbin], color="orange", label="Largest positive errors"
) 
plt.xlabel("Actuator ID")
plt.ylabel("Hits per slew")
#plt.yscale('log')
plt.grid(axis="y",which="minor")
plt.title(f"Actuator IDs with largest positive primary Force Actuator Errors in a slew {day_obs}")
plt.legend()

In [ ]:
minhist = plt.hist(
    min_actuator_ids_primary, bins=maxbin-minbin, range=[minbin, maxbin], color="red", label="Largest negative errors"
)
plt.xlabel("Actuator ID")
plt.ylabel("Hits per slew")
#plt.yscale('log')
plt.grid(axis="y",which="minor")
plt.title(f"Actuator IDs with largest negative primary Force Actuator Errors in a slew {day_obs}")
plt.legend()

In [ ]:
maxhist = plt.hist(
    max_actuator_ids_secondary, bins=maxbin-minbin, range=[minbin, maxbin], color="orange", label="Largest positive errors"
) 
plt.xlabel("Actuator ID")
plt.ylabel("Hits per slew")
#plt.yscale('log')
plt.grid(axis="y",which="minor")
plt.title(f"Actuator IDs with largest positive secondary Force Actuator Errors in a slew {day_obs}")
plt.legend()

In [ ]:
minhist = plt.hist(
    min_actuator_ids_secondary, bins=maxbin-minbin, range=[minbin, maxbin], color="red", label="Largest negative errors"
)
plt.xlabel("Actuator ID")
plt.ylabel("Hits per slew")
#plt.yscale('log')
plt.grid(axis="y",which="minor")
plt.title(f"Actuator IDs with largest negative secondary Force Actuator Errors in a slew {day_obs}")
plt.legend()

### Compute FFT for FA errors in the time range selected above

In [ ]:
df = getEfdData(
    client,"lsst.sal.MTM1M3.forceActuatorData", 
    columns=primary_FA_error, 
    prePadding = pre_slew_delta,
    postPadding = post_slew_delta,
    begin=t_start_freq_analysis, 
    end=t_end_freq_analysis,
    #begin=slews[-1].begin, 
    #end=slews[-1].end,
)
dt = (df['primaryCylinderFollowingError0'].index[1] -
        df['primaryCylinderFollowingError0'].index[0]).total_seconds()
freqs = np.fft.fftfreq(len(df), d=dt)
positive_mask = freqs > 0
fft_frequency = freqs[positive_mask]
fft_result = np.fft.fft(df.values, axis=0)
fft_magnitudes = np.abs(fft_result[positive_mask, :])
peak_freqs_histo = []
for j in range(N_ACTUATORS):
    peaks, properties = find_peaks(fft_magnitudes[:, j], height=10, distance=5)
    # Extract peak frequencies and magnitudes
    peak_freqs = fft_frequency[peaks]
    peak_magnitudes = fft_magnitudes[peaks, j]
    largest = pd.Series(peak_magnitudes).nlargest(5)
    for k in largest.index:
        peak_freqs_histo.append(peak_freqs[k]) 
#print(peak_freqs_histo)
plt.figure(figsize=(10, 5))
plt.plot(fft_frequency, fft_magnitudes)
plt.title("FFT Magnitude Spectrum")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Magnitude")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(peak_freqs_histo, bins=30)
plt.title(f"Top frequencies in the range from {t_start_freq_analysis.strftime('%Y-%m-%d %H:%M:%S')} to {t_end_freq_analysis.strftime('%Y-%m-%d %H:%M:%S')}")
plt.xlabel("Frequency [Hz]")